## Importing Required Libraries.

In [1]:
import os
import json
import random
import glob
from pathlib import Path

import numpy as np
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from facenet_pytorch import MTCNN

from transformers import ViTModel
from transformers import get_cosine_schedule_with_warmup

from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score
)

from tqdm.notebook import tqdm
print("Done")

Done


In [2]:
# Compute using GPU not CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active processing device: {device}")

Active processing device: mps


## **Step 1.1:** Parse the `metadata.json` files for DFDC Parts 0-4.

In [3]:
def parse_dfdc_metadata(base_dir, parts_to_parse=None):
    """
    Step 1.1: Parses metadata.json files from all available DFDC part folders.
    
    Args:
        base_dir (str): The root directory containing the part folders 
                        (e.g., 'dfdc_train_part_0', 'dfdc_train_part_1', etc.).
        parts_to_parse (list/range, optional): Explicit part numbers to parse.
                        If omitted, all matching part folders under base_dir are used.
        
    Returns:
        dict: An aggregated dictionary of all metadata, keeping track of which 
              folder each video belongs to.
    """
    aggregated_metadata = {}
    missing_parts = []

    if parts_to_parse is None:
        available_parts = sorted(
            [
                name for name in os.listdir(base_dir)
                if name.startswith("dfdc_train_part_") and os.path.isdir(os.path.join(base_dir, name))
            ]
        )
        parts_to_parse = [int(name.split("_")[-1]) for name in available_parts]

    parts_to_parse = list(parts_to_parse)
    print(f"Scanning base directory: {base_dir} for parts {parts_to_parse}...")

    for part in parts_to_parse:
        folder_name = f"dfdc_train_part_{part}"
        json_path = os.path.join(base_dir, folder_name, "metadata.json")
        
        if not os.path.exists(json_path):
            print(f"Warning: Could not find {json_path}")
            missing_parts.append(part)
            continue
            
        with open(json_path, 'r') as f:
            try:
                part_metadata = json.load(f)
                for video_name, data in part_metadata.items():
                    data['part_folder'] = folder_name
                    aggregated_metadata[video_name] = data
                    
            except json.JSONDecodeError:
                print(f"❌ Error: {json_path} is corrupted or not a valid JSON file.")
                
    print(f"✅ Successfully parsed {len(aggregated_metadata)} total video records.")
    if missing_parts:
        print(f"⚠️ Note: Missing part folders: {missing_parts}. Ensure your paths are correct.")
        
    return aggregated_metadata

# ==========================================
# How to run it in your notebook:
# ==========================================
# # Set this to the folder that CONTAINS your 'dfdc_train_part_X' folders
# ROOT_DATASET_DIR = "path/to/your/dataset_parent_folder" 

# # # Output: {'label': 'FAKE', 'split': 'train', 'original': 'vudstovrck.mp4', 'part_folder': 'dfdc_train_part_0'}

# all_metadata = parse_dfdc_metadata(ROOT_DATASET_DIR, parts_to_parse=[0, 1, 2, 3, 4])# # print(all_metadata['aagfhgtpmv.mp4'])

# # # Example of what one record looks like now:

## **Step 1.2:** Write a balancing script to isolate an equal count of REAL and FAKE video IDs.

In [5]:
import os
import json


def parse_dfdc_metadata(base_dir, parts_to_parse=None):
    aggregated_metadata = {}
    missing_parts = []

    if parts_to_parse is None:
        available_parts = sorted(
            [
                name for name in os.listdir(base_dir)
                if name.startswith("dfdc_train_part_") and os.path.isdir(os.path.join(base_dir, name))
            ]
        )
        parts_to_parse = [int(name.split("_")[-1]) for name in available_parts]

    for part in parts_to_parse:
        folder_name = f"dfdc_train_part_{part}"
        json_path = os.path.join(base_dir, folder_name, "metadata.json")

        if not os.path.exists(json_path):
            print(f"Warning: Could not find {json_path}")
            missing_parts.append(part)
            continue

        with open(json_path, "r") as f:
            part_metadata = json.load(f)
            for video_name, data in part_metadata.items():
                data["part_folder"] = folder_name
                aggregated_metadata[video_name] = data

    return aggregated_metadata

In [6]:
import os
import json
import random


def parse_dfdc_metadata(base_dir, parts_to_parse=None):
    aggregated_metadata = {}
    missing_parts = []

    if parts_to_parse is None:
        available_parts = sorted(
            [
                name for name in os.listdir(base_dir)
                if name.startswith("dfdc_train_part_") and os.path.isdir(os.path.join(base_dir, name))
            ]
        )
        parts_to_parse = [int(name.split("_")[-1]) for name in available_parts]

    for part in parts_to_parse:
        folder_name = f"dfdc_train_part_{part}"
        json_path = os.path.join(base_dir, folder_name, "metadata.json")

        if not os.path.exists(json_path):
            print(f"Warning: Could not find {json_path}")
            missing_parts.append(part)
            continue

        with open(json_path, "r") as f:
            part_metadata = json.load(f)
            for video_name, data in part_metadata.items():
                data["part_folder"] = folder_name
                aggregated_metadata[video_name] = data

    print(f"✅ Parsed {len(aggregated_metadata)} records")
    if missing_parts:
        print(f"⚠️ Missing parts: {missing_parts}")
    return aggregated_metadata


def balance_dataset(all_metadata, random_seed=42):
    random.seed(random_seed)
    real_videos = []
    fake_videos = []

    for video_name, data in all_metadata.items():
        if data.get("label") == "REAL":
            real_videos.append(video_name)
        elif data.get("label") == "FAKE":
            fake_videos.append(video_name)

    print(f"📊 Initial Dataset Skew -> REAL: {len(real_videos)} | FAKE: {len(fake_videos)}")

    if len(fake_videos) > len(real_videos):
        balanced_fake = random.sample(fake_videos, len(real_videos))
        balanced_real = real_videos
    elif len(real_videos) > len(fake_videos):
        balanced_real = random.sample(real_videos, len(fake_videos))
        balanced_fake = fake_videos
    else:
        balanced_real = real_videos
        balanced_fake = fake_videos

    target_videos = balanced_real + balanced_fake
    random.shuffle(target_videos)

    print(f"⚖️ Final Balanced Count -> REAL: {len(balanced_real)} | FAKE: {len(balanced_fake)}")
    print(f"✅ Total videos queued for extraction: {len(target_videos)}")
    return target_videos


ROOT_DATASET_DIR = "/Users/jellyfish/DeveloperLocal/DeepFake_Detection_ViViT/dfdc_train_dataset"
all_metadata = parse_dfdc_metadata(ROOT_DATASET_DIR)
print(f"Metadata entries: {len(all_metadata)}")
print(f"Available part folders: {sorted([name for name in os.listdir(ROOT_DATASET_DIR) if name.startswith('dfdc_train_part_')])}")
balanced_video_list = balance_dataset(all_metadata)
print(f"First 10 balanced videos: {balanced_video_list[:10]}")

✅ Parsed 13884 records
Metadata entries: 13884
Available part folders: ['dfdc_train_part_0', 'dfdc_train_part_1', 'dfdc_train_part_2', 'dfdc_train_part_3', 'dfdc_train_part_4', 'dfdc_train_part_5', 'dfdc_train_part_6']
📊 Initial Dataset Skew -> REAL: 1609 | FAKE: 12275
⚖️ Final Balanced Count -> REAL: 1609 | FAKE: 1609
✅ Total videos queued for extraction: 3218
First 10 balanced videos: ['klmaeumvul.mp4', 'ntkrwyqoop.mp4', 'eneofiltbk.mp4', 'cruebsmlgr.mp4', 'gxfaiqipcw.mp4', 'jpvkoxrueh.mp4', 'tblcswchep.mp4', 'cixupjraoc.mp4', 'eahthbqnbe.mp4', 'eerhmzaebh.mp4']


## **Step 1.3:** Develop the OpenCV video ingestion pipeline to sample 5 to 10 frames per target video.

## **Step 1.4:** Implement MTCNN for facial detection on the sampled frames.

## **Step 1.5:** Apply the bounding box expansion logic (margin addition) and save the cropped faces as `.jpg` files organized into `train/REAL`, `train/FAKE`, `val/REAL`, and `val/FAKE` directories based on the Video-ID split rule.
(371 min)

In [ ]:
# Jupyter cell: Video ingestion, MTCNN detection, bbox expansion, and saving crops
# Usage: adjust `VIDEO_ROOT`, `OUTPUT_ROOT`, `NUM_FRAMES`, `MARGIN`, `TRAIN_SPLIT_RATIO` then run.
import os
import glob
import json
import hashlib
from pathlib import Path
from tqdm.notebook import tqdm

import cv2
import numpy as np
from PIL import Image
import torch
from facenet_pytorch import MTCNN

# Config
VIDEO_ROOT = "/Users/jellyfish/DeveloperLocal/DeepFake_Detection_ViViT/dfdc_train_dataset"   # root containing dfdc_train_part_*/metadata.json and videos
OUTPUT_ROOT = "/Users/jellyfish/DeveloperLocal/DeepFake_Detection_ViViT/processed_faces_v2"     # will create train/REAL, train/FAKE, val/REAL, val/FAKE
NUM_FRAMES = 15                     
MARGIN = 0.30                       
TRAIN_SPLIT_RATIO = 0.8             
USE_LARGEST_FACE_ONLY = True        
MIN_FACE_SIZE = 60                  
REQUIRE_ALL_FRAMES_VALID = True     

# Device for MTCNN
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# Make MPS use the fastest available settings on Apple Silicon
if device == "mps":
    torch.mps.set_per_process_memory_fraction(0.9)
    torch.mps.set_per_process_memory_fraction(0.9)

# NOTE: facenet-pytorch MTCNN/resize path can trigger unsupported adaptive pooling on MPS.
# Workaround: run detection on CPU when running on MPS to avoid the runtime error.
mtcnn_device = "cpu" if device == "mps" else device
print(f"Building MTCNN on device: {mtcnn_device} (runtime device: {device})")
mtcnn = MTCNN(keep_all=True, device=mtcnn_device)


def read_all_metadata(root):
    mapping = {}
    for meta_path in glob.glob(os.path.join(root, "dfdc_train_part_*", "metadata.json")):
        with open(meta_path, "r") as f:
            part_map = json.load(f)
        mapping.update(part_map)
    return mapping


def find_video_file(root, filename):
    matches = glob.glob(os.path.join(root, "**", filename), recursive=True)
    return matches[0] if matches else None


def deterministic_split(video_id, train_ratio=0.8):
    h = int(hashlib.md5(video_id.encode("utf-8")).hexdigest()[:8], 16)
    return "train" if (h % 1000) / 1000.0 < train_ratio else "val"


def sample_frame_indices(frame_count, num_samples):
    if frame_count <= num_samples:
        return list(range(frame_count))
    positions = np.linspace(0, frame_count - 1, num_samples, dtype=int)
    return list(np.unique(positions))


def expand_bbox(box, img_w, img_h, margin):
    x1, y1, x2, y2 = box
    w = x2 - x1
    h = y2 - y1
    x1n = max(0, int(x1 - margin * w))
    y1n = max(0, int(y1 - margin * h))
    x2n = min(img_w, int(x2 + margin * w))
    y2n = min(img_h, int(y2 + margin * h))
    return x1n, y1n, x2n, y2n


def process_video(video_path, video_filename, label, out_root, mtcnn, num_frames=7, margin=0.30):
    split = deterministic_split(video_filename, TRAIN_SPLIT_RATIO)
    out_dir = os.path.join(out_root, split, label)
    os.makedirs(out_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return 0

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if frame_count == 0:
        cap.release()
        return 0

    indices = sample_frame_indices(frame_count, num_samples=num_frames)
    saved = 0
    valid_frame_count = 0

    for idx in tqdm(indices, desc=f"Frames {Path(video_filename).name}", leave=False):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret or frame is None:
            continue

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)

        boxes, probs = mtcnn.detect(pil_img)
        if boxes is None:
            continue

        valid_boxes = []
        for box in boxes:
            if box is None:
                continue
            x1, y1, x2, y2 = box
            width = int(x2 - x1)
            height = int(y2 - y1)
            if width < MIN_FACE_SIZE or height < MIN_FACE_SIZE:
                continue
            valid_boxes.append(box)

        if not valid_boxes:
            continue

        valid_frame_count += 1

        if USE_LARGEST_FACE_ONLY and len(valid_boxes) > 1:
            areas = [(b[2]-b[0])*(b[3]-b[1]) for b in valid_boxes]
            idx_largest = int(np.argmax(areas))
            faces_to_use = [valid_boxes[idx_largest]]
        else:
            faces_to_use = valid_boxes

        for fi, box in enumerate(faces_to_use):
            x1, y1, x2, y2 = box
            img_w, img_h = pil_img.width, pil_img.height
            x1n, y1n, x2n, y2n = expand_bbox((x1, y1, x2, y2), img_w, img_h, margin)
            crop = pil_img.crop((x1n, y1n, x2n, y2n))
            base = os.path.splitext(video_filename)[0]
            out_name = f"{base}_f{idx:06d}"
            if len(faces_to_use) > 1:
                out_name += f"_p{fi}"
            out_path = os.path.join(out_dir, out_name + ".jpg")
            crop.save(out_path, quality=95)
            saved += 1

    cap.release()

    if REQUIRE_ALL_FRAMES_VALID and valid_frame_count != len(indices):
        # Remove all newly saved images for this video if not every sampled frame passed validation.
        for file_name in os.listdir(out_dir):
            if file_name.startswith(os.path.splitext(video_filename)[0] + "_f"):
                os.remove(os.path.join(out_dir, file_name))
        return 0

    return saved


def main(video_root=VIDEO_ROOT, output_root=OUTPUT_ROOT, num_frames=NUM_FRAMES, margin=MARGIN):
    import time
    metadata = read_all_metadata(video_root)

    if "balanced_video_list" in globals() and balanced_video_list:
        video_ids_to_process = balanced_video_list
        print(f"Using balanced subset of {len(video_ids_to_process)} videos")
    else:
        video_ids_to_process = list(metadata.keys())
        print("balanced_video_list is not available; processing all metadata entries")

    print(f"Found metadata entries: {len(metadata)}")
    total_saved = 0
    errors = []
    processed = 0

    for idx, video_filename in enumerate(tqdm(video_ids_to_process, desc="Videos", unit="video"), 1):
        try:
            info = metadata.get(video_filename, {})
            label = str(info.get("label", "FAKE")).upper()
            if label not in ("REAL", "FAKE"):
                if str(label).lower() in ("0", "fake", "f"):
                    label = "FAKE"
                else:
                    label = "REAL"

            video_path = find_video_file(video_root, video_filename)
            if not video_path:
                continue

            saved = process_video(video_path, video_filename, label, output_root, mtcnn, num_frames, margin)
            total_saved += saved
            processed += 1
            
            if idx % 100 == 0:
                print(f"\n✓ Processed {idx}/{len(video_ids_to_process)} videos | Saved {total_saved} faces")
                
        except Exception as e:
            errors.append((video_filename, str(e)))
            if len(errors) <= 5:
                print(f"\n⚠️ Error processing {video_filename}: {str(e)[:100]}")
            continue

    print(f"\n{'='*60}")
    print(f"Finished. Total videos processed: {processed}")
    print(f"Total face crops saved: {total_saved}")
    if errors:
        print(f"Errors encountered: {len(errors)}")
        print(f"First few errors: {errors[:3]}")


# Run
if __name__ == "__main__":
    main()


Check the count 

In [36]:
import os

output_root = "/Users/jellyfish/DeveloperLocal/DeepFake_Detection_ViViT/processed_faces_v2"
for split in ["train", "val"]:
    for label in ["REAL", "FAKE"]:
        folder = os.path.join(output_root, split, label)
        count = 0
        if os.path.isdir(folder):
            count = len([f for f in os.listdir(folder) if f.lower().endswith(".jpg")])
        print(f"{split}/{label}: {count}")

train/REAL: 0
train/FAKE: 0
val/REAL: 0
val/FAKE: 0
